In [13]:
import pandas as pd
import numpy as np

In [14]:
prov_avai = pd.read_csv('../data2/environment_daily.csv')
combi = prov_avai[['province', 'amphoe', 'tambon']].drop_duplicates()
combi

,province,amphoe,tambon
0,น่าน,เฉลิมพระเกียรติ,ห้วยโก๋น
1,พะเยา,จุน,ห้วยข้าวก่ำ
2,พะเยา,เชียงม่วน,เชียงม่วน
3,ลำปาง,แม่เมาะ,แม่เมาะ
4,ลำพูน,ลี้,ลี้
5,เชียงราย,ป่าแดด,ป่าแดด
6,เชียงราย,เทิง,เวียง
7,เชียงราย,เมือง,รอบเวียง
8,เชียงราย,แม่สาย,เวียงพางคำ
9,เชียงใหม่,ดอยเต่า,ดอยเต่า


In [15]:
patient = pd.read_csv('../data2/patients.csv')
print(patient.columns)
patient.iloc[0].to_dict()

Index(['patient_id', 'name_masked', 'sex', 'age', 'province', 'amphoe',
       'tambon', 'last_contact_date'],
      dtype='object')


{'patient_id': 'PT000005',
 'name_masked': 'xxx-2290',
 'sex': 1,
 'age': 33,
 'province': 'เชียงราย',
 'amphoe': 'แม่สาย',
 'tambon': 'เวียงพางคำ',
 'last_contact_date': '2026-05-01'}

In [16]:
is_mueang = combi['amphoe'].str.contains('เมือง', na=False)

peak_weight = 15.0  
tail_weight = 1.0   

# Apply the weights to the dataset
combi['sample_weight'] = np.where(is_mueang, peak_weight, tail_weight)

# 3. Sample 10,000 patients using the custom probability distribution
randomized_addresses = combi.sample(
    n=10000, 
    replace=True, 
    weights=combi['sample_weight'],
    random_state=42 # Set a seed so your mock data is reproducible
).reset_index(drop=True)

# Clean up the helper column so it doesn't end up in your final CSV
randomized_addresses = randomized_addresses.drop(columns=['sample_weight'])

# You can check the distribution ratio quickly like this:
print(randomized_addresses['amphoe'].str.contains('เมือง').value_counts(normalize=True))

amphoe
False    0.5986
True     0.4014
Name: proportion, dtype: float64


In [17]:
num_records = len(randomized_addresses)
mock_patients = randomized_addresses.copy()
mock_patients['patient_id'] = [f"PT{str(i).zfill(6)}" for i in range(1, num_records + 1)]
mock_patients['name_masked'] = [f"xxx-{np.random.randint(1000, 9999)}" for _ in range(num_records)]
mock_patients['sex'] = np.random.choice([1, 2], size=num_records)
mock_patients['age'] = np.random.randint(0, 96, size=num_records)
date_range = pd.date_range(start=patient['last_contact_date'].min(), end=patient['last_contact_date'].max()).strftime('%Y-%m-%d')
mock_patients['last_contact_date'] = np.random.choice(date_range, size=num_records)
expected_columns = patient.columns
mock_patients = mock_patients[expected_columns]

display(mock_patients.head())
print("\nSample Dict Output:")
print(mock_patients.iloc[0].to_dict())

,patient_id,name_masked,sex,age,province,amphoe,tambon,last_contact_date
0,PT000001,xxx-9451,2,22,เชียงราย,เมือง,รอบเวียง,2026-05-06
1,PT000002,xxx-7953,1,29,ลำปาง,วังเหนือ,วังเหนือ,2026-05-14
2,PT000003,xxx-4575,1,62,เชียงใหม่,แม่วาง,บ้านกาด,2026-04-21
3,PT000004,xxx-1331,1,8,เชียงราย,แม่สาย,เวียงพางคำ,2026-05-12
4,PT000005,xxx-1495,2,74,เชียงราย,ป่าแดด,ป่าแดด,2026-04-22



Sample Dict Output:
{'patient_id': 'PT000001', 'name_masked': 'xxx-9451', 'sex': 2, 'age': 22, 'province': 'เชียงราย', 'amphoe': 'เมือง', 'tambon': 'รอบเวียง', 'last_contact_date': '2026-05-06'}


In [18]:
mock_patients.to_csv('../data2/patients.csv', index = False)